# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ijazkhan0351-bot/Ijazweek1-ml-assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

I'm using **Logistic Regression** as my primary model, with a **Random
Forest** as a comparison.

Why: My baseline (Week 4) is a hand-written multiplicative rule — it can't
learn interactions between signals or weight them optimally. Logistic
Regression is the natural next step: it's interpretable (I can read the
coefficients), handles a binary "declining vs not" target well, and won't
overfit on a modest feature set. Random Forest adds the ability to capture
non-linear interactions between signals — comparing both tells me whether
the extra complexity of Random Forest is actually earning its keep.

I am NOT using Gradient Boosting this week — with a modest feature set, it
risks overfitting without adding real signal.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

# This cell is for CODE (numbers, a query, a check).## Split Design

**Grouped, time-aware split.** I define my label by comparing March
(month=2026-03, features) against April (month=2026-04, outcome) — a
genuine past→future setup, never touching the sealed June test month.

I also split by `client_hash_id` (GroupShuffleSplit), not by row, so that
pages from the same client never appear in both train and test.
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
march = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(gsc_impressions) as avg_impressions,
        AVG(gsc_clicks) as avg_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

april = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) as april_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

data = march.merge(april, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["april_clicks"] < data["avg_clicks"] * 30 * 0.85).astype(int)

print(f"Total pages with both months' data: {len(data)}")
print(f"Declining rate: {data['is_declining'].mean():.3f}")

feature_cols = ["avg_impressions", "avg_clicks", "avg_position", "ctr"]
X = data[feature_cols].fillna(0)
y = data["is_declining"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
data_test = data.iloc[test_idx].copy()

print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")
print(f"Client overlap between train/test: {len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

vol_norm = (data_test["avg_impressions"] - data["avg_impressions"].min()) / \
           (data["avg_impressions"].max() - data["avg_impressions"].min())
ctr_gap = 1 - (data_test["ctr"] / data["ctr"].max())
baseline_score = vol_norm * ctr_gap
baseline_p50 = precision_at_k(baseline_score, y_test, 50)

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(logreg_score, y_test, 50)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_score, y_test, 50)

print("=== Model vs Baseline (Precision@50, same test split) ===")
print(f"Week-4 Baseline (hand rule):  {baseline_p50:.3f}")
print(f"Logistic Regression:          {logreg_p50:.3f}")
print(f"Random Forest:                {rf_p50:.3f}")

## Model vs Baseline

[Yahan real table numbers daalenge run hone ke baad]

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Logistic Regression coefficients ===")
for feat, coef in zip(feature_cols, logreg.coef_[0]):
    print(f"{feat}: {coef:.3f}")

print("\n=== Random Forest feature importance ===")
for feat, imp in zip(feature_cols, rf.feature_importances_):
    print(f"{feat}: {imp:.3f}")

data_test["rf_score"] = rf_score
data_test["true_label"] = y_test.values
errors = data_test.sort_values("rf_score", ascending=False).head(50)
false_positives = errors[errors["true_label"] == 0]
print(f"\nFalse positives in top 50: {len(false_positives)}")

## Errors and Interpretation

[Yahan real observations likhenge output dekhne ke baad]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.